# Computing 2D Point Spread Functions

This notebook demonstrates the computation of 2D PSFs at the focal plane and examines the influence of optical parameters on system resolution.

## Import Libraries

In [ ]:
from psf_generator.propagators import ScalarSphericalPropagator
from psf_generator.utils.plots import plot_psf

## Example 1: Computing a 2D PSF

We compute the PSF at the focal plane using the `ScalarSphericalPropagator`, which treats light as a scalar wave. This approximation is valid for moderate numerical apertures (NA < 0.9).

### System Parameters

The optical system parameters are:
- `n_pix_pupil`: Pupil plane discretization
- `n_pix_psf`: Image plane pixel count
- `na`: Numerical aperture
- `wavelength`: Wavelength (nm)
- `pix_size`: Pixel size (nm)

In [ ]:
base_kwargs = {
    'n_pix_pupil': 127,
    'n_pix_psf': 256,
    'na': 0.8,
    'wavelength': 632,
    'pix_size': 10,
}

### PSF Computation

In [ ]:
propagator = ScalarSphericalPropagator(**base_kwargs)
psf = propagator.compute_focus_field()

### Visualization

The PSF is a complex-valued field. We visualize its modulus (|E|), phase (arg(E)), and intensity (|E|²).

In [ ]:
for quantity in ['modulus', 'phase', 'intensity']:
    plot_psf(
        psf=psf,
        name_of_propagator=propagator.get_name(),
        quantity=quantity,
        show_titles=True,
        show_cbar_ticks=True
    )

## Example 2: Effect of Physical Parameters on Resolution

According to the Abbe diffraction limit, the lateral resolution is $d = \lambda/(2\mathrm{NA})$. Higher resolution is achieved by increasing NA or decreasing wavelength.

In [ ]:
high_res_kwargs = {
    **base_kwargs,
    'na': 1.3,
    'wavelength': 480,
}

### Computation and Comparison

In [ ]:
high_res_propagator = ScalarSphericalPropagator(**high_res_kwargs)
high_res_psf = high_res_propagator.compute_focus_field()

# High-resolution system
plot_psf(
    psf=high_res_psf,
    name_of_propagator=high_res_propagator.get_name(),
    quantity='intensity',
    show_titles=True,
    show_cbar_ticks=True
)

# Original system for comparison
plot_psf(
    psf=psf,
    name_of_propagator=propagator.get_name(),
    quantity='intensity',
    show_titles=True,
    show_cbar_ticks=True
)

The higher NA and shorter wavelength system produces a more tightly confined PSF, indicating improved lateral resolution (d ≈ 185 nm vs. 395 nm for the original system).

## Troubleshooting

**Performance optimization:**
- Reduce `n_pix_pupil` for faster computation (at the cost of accuracy)
- Add `'device': 'cuda:0'` for GPU acceleration

**Accuracy considerations:**
- Increase `n_pix_pupil` (127, 257, or 513) for better numerical convergence
- Ensure `pix_size` adequately samples the PSF (typically 5-20 nm)

**Memory constraints:**
- Reduce `n_pix_psf` to decrease memory footprint

**High NA systems (NA > 0.9):**
- Use vectorial propagators (`VectorialSphericalPropagator` or `VectorialCartesianPropagator`) for accurate field representation